# Baseline ETA taxi NYC (notebook của team DS)

Notebook này là **điểm xuất phát** của capstone: một bạn DS viết để trả lời câu "dự đoán thời gian chuyến đi có
khả thi không". Nó chạy được, cho ra số, nhưng chưa vận hành được: đường dẫn cứng, luật làm sạch nằm rải rác, không có
config, không có test, không ghi lại model nào được train trên dữ liệu nào.

Việc của bạn ở M01 là biến nó thành package `eta` (xem `../../milestones/m01-package.md`). **Đừng sửa notebook này
để làm bài**; giữ nó làm mốc so sánh. Chạy hết vài giây trên laptop (đọc 5 cột của một tháng, lấy mẫu 200 000 dòng).

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.ensemble import HistGradientBoostingRegressor

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "nyc-tlc").is_dir())
PATH = ROOT / "data" / "raw" / "nyc-tlc" / "yellow_tripdata_2019-01.parquet"
t0 = time.time()

## Đọc dữ liệu

Tệp tháng 1/2019 có khoảng 7,7 triệu chuyến và 19 cột. Chỉ đọc 5 cột cần, rồi lấy mẫu 200 000 dòng bằng
`numpy.random.default_rng(0)` để lần nào chạy cũng ra cùng mẫu.

In [2]:
cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance"]
table = pq.read_table(PATH, columns=cols)
idx = np.sort(np.random.default_rng(0).choice(table.num_rows, size=200_000, replace=False))
df = table.take(idx).to_pandas()
print(table.num_rows, "chuyến trong tháng, lấy mẫu", len(df))
df.head()

7696617 chuyến trong tháng, lấy mẫu 200000


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance
0,2019-01-01 00:50:06,2019-01-01 01:11:36,107,181,4.6
1,2019-01-01 00:32:59,2019-01-01 00:32:59,237,264,0.0
2,2019-01-01 00:18:09,2019-01-01 00:24:34,144,249,1.0
3,2019-01-01 00:50:41,2019-01-01 01:00:04,264,264,1.3
4,2019-01-01 00:14:27,2019-01-01 00:16:47,141,263,0.5


## Dữ liệu bẩn

Đếm từng loại bất thường trên mẫu. Một dòng có thể dính nhiều loại.

In [3]:
df["duration_s"] = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds()
issues = {
    "thời lượng < 60 s (kể cả âm, 0)": (df.duration_s < 60).sum(),
    "thời lượng > 3 giờ": (df.duration_s > 10_800).sum(),
    "trip_distance <= 0": (df.trip_distance <= 0).sum(),
    "trip_distance > 100 dặm": (df.trip_distance > 100).sum(),
    "pickup ngoài tháng 1/2019": (~df.tpep_pickup_datetime.between("2019-01-01", "2019-01-31 23:59:59.999999")).sum(),
    "zone 264/265 (Unknown)": ((df.PULocationID > 263) | (df.DOLocationID > 263)).sum(),
}
pd.Series(issues, name="số dòng")

thời lượng < 60 s (kể cả âm, 0)    1956
thời lượng > 3 giờ                  539
trip_distance <= 0                 1483
trip_distance > 100 dặm               1
pickup ngoài tháng 1/2019            11
zone 264/265 (Unknown)             4713
Name: số dòng, dtype: int64

In [4]:
df = df[(df.duration_s >= 60) & (df.duration_s <= 10800)]
df = df[(df.trip_distance > 0) & (df.trip_distance <= 100)]
df = df[(df.tpep_pickup_datetime >= "2019-01-01") & (df.tpep_pickup_datetime < "2019-02-01")]
df = df[(df.PULocationID <= 263) & (df.DOLocationID <= 263)]
print("còn", len(df), "dòng sau làm sạch")

còn 192918 dòng sau làm sạch


## Chia train/test theo thời gian

20% chuyến cuối tháng làm test. Chia ngẫu nhiên sẽ cho số đẹp hơn thật vì model "nhìn thấy" cùng ngày, cùng trận
tuyết, cùng sự kiện với chuyến cần đoán.

In [5]:
df = df.sort_values("tpep_pickup_datetime", kind="mergesort").reset_index(drop=True)
cut = int(len(df) * 0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
print(train.tpep_pickup_datetime.max(), "|", test.tpep_pickup_datetime.min())

2019-01-26 00:40:03 | 2019-01-26 00:40:18


## Baseline: trung vị theo cặp zone

Đoán thời gian chuyến bằng trung vị thời gian của các chuyến cùng cặp (zone đón, zone trả) trong tập train; cặp
chưa gặp thì dùng trung vị toàn bộ. Đây là mốc mà mọi model phải thắng.

In [6]:
med = train.groupby(["PULocationID", "DOLocationID"]).duration_s.median().rename("pred")
pred = test.join(med, on=["PULocationID", "DOLocationID"]).pred.fillna(train.duration_s.median())
baseline_mae = float(np.abs(test.duration_s - pred).mean())
print(f"MAE baseline trung vị theo cặp zone: {baseline_mae:.1f} s")

MAE baseline trung vị theo cặp zone: 243.0 s


## Model thô

Gradient boosting với zone để nguyên dạng số (zone 132 không "lớn hơn" zone 48, nhưng cứ thử), thêm giờ và thứ
trong tuần.

In [7]:
def feats(d):
    return pd.DataFrame({
        "PULocationID": d.PULocationID, "DOLocationID": d.DOLocationID,
        "hour": d.tpep_pickup_datetime.dt.hour, "dow": d.tpep_pickup_datetime.dt.day_of_week,
        "trip_distance": d.trip_distance,
    })

model = HistGradientBoostingRegressor(max_iter=200, random_state=0).fit(feats(train), train.duration_s)
err = np.abs(test.duration_s - model.predict(feats(test)))
airport = test.PULocationID.isin([132, 138]) | test.DOLocationID.isin([132, 138])
print(f"MAE model thô: {err.mean():.1f} s | p90 sai số: {np.quantile(err, 0.9):.1f} s | MAE chuyến sân bay: {err[airport].mean():.1f} s")
print(f"Chạy hết {time.time() - t0:.0f} s")

MAE model thô: 158.8 s | p90 sai số: 353.2 s | MAE chuyến sân bay: 302.0 s
Chạy hết 1 s


## Notebook này thiếu gì để lên production

- Đường dẫn và tham số (tháng, cỡ mẫu, seed, hyperparameter) nằm cứng trong code.
- Luật làm sạch viết bốn dòng lọc liên tiếp, không ai kiểm được "còn bao nhiêu dòng là đúng".
- Không có hàm nào để test; chạy lại cả notebook là cách kiểm duy nhất.
- Model không được lưu, không biết model nào train trên dữ liệu nào, commit nào.
- Zone là mã danh mục nhưng đang dùng như số.
- Serving dựng lại feature thế nào? Nếu viết lại `feats()` ở chỗ khác là có nguy cơ training-serving skew.

M01 → M04 lần lượt xử lý từng điểm này.